In [ ]:
#!pip install rdflib networkx torch torch-geometric matplotlib

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx

from rdflib.plugins.sparql import parser, algebra
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1) Example SPARQL Triples for Contrastive Learning

In [2]:
# ------------------------------------------------------------
#    - anchor_query
#    - positive_queries (list)
#    - negative_queries (list)
# ------------------------------------------------------------

anchor_query = """
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?person WHERE {
  ?person dbo:birthPlace dbo:Berlin .
  ?person dbo:occupation ?job .
  FILTER regex(?job, "actor")
}
"""

positive_queries = [
"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?p WHERE {
  ?p dbo:birthPlace dbo:Berlin ;
     dbo:occupation ?j .
  FILTER regex(?j, "actor")
}
""",
"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?person WHERE {
  ?person dbo:birthPlace dbo:Berlin .
  ?person dbo:occupation ?o .
  FILTER regex(?o, "actor")
}
"""
]

negative_queries = [
"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?film WHERE {
  ?film dbo:director ?d .
  ?film dbo:starring ?actor .
}
""",
"""
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?city WHERE {
  ?city dbo:country dbo:Germany .
}
"""
]

print("Anchor:\n", anchor_query)
print("Num positives:", len(positive_queries))
print("Num negatives:", len(negative_queries))


Anchor:
 
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?person WHERE {
  ?person dbo:birthPlace dbo:Berlin .
  ?person dbo:occupation ?job .
  FILTER regex(?job, "actor")
}

Num positives: 2
Num negatives: 2


 ------------------------------------------------------------
### 2) Convert SPARQL Algebra → NetworkX Graph
 ------------------------------------------------------------


In [3]:
def algebra_to_graph(op, G=None, parent=None, node_id=0):
    """
    Recursively builds a directed graph from the SPARQL algebra tree.
    Each algebra operator becomes a node, and BGP triples become leaf nodes.
    """
    if G is None:
        G = nx.DiGraph()

    # Label of the current algebra operator
    node_label = op.name if hasattr(op, 'name') else type(op).__name__
    node_name = f"{node_label}_{node_id}"

    G.add_node(node_name, label=node_label)

    if parent is not None:
        G.add_edge(parent, node_name)

    # Special case: Basic Graph Pattern (BGP) triples
    if getattr(op, "name", None) == "BGP":
        for s, p, o in op.triples:
            triple_node = f"Triple_{s}_{p}_{o}_{node_id}"
            triple_label = f"{s} {p} {o}"
            G.add_node(triple_node, label=triple_label)
            G.add_edge(node_name, triple_node)

    # Recurse over attributes/values in the algebra object
    # to find nested algebra operations
    for v in op.values():
        if hasattr(v, "name"):  # nested algebra op
            node_id += 1
            G = algebra_to_graph(v, G, node_name, node_id)
        elif isinstance(v, list):
            for sub in v:
                if hasattr(sub, "name"):
                    node_id += 1
                    G = algebra_to_graph(sub, G, node_name, node_id)

    return G


### 3) Convert NetworkX → PyTorch Geometric Data

In [4]:
def graph_to_pyg(G):
    labels = nx.get_node_attributes(G, "label")

    # Simple label hashing → 3D feature vector per node
    feats = []
    for lbl in labels.values():
        h = abs(hash(lbl)) % (10**6)
        vec = np.array([
            h % 97,
            (h // 97) % 97,
            (h // (97**2)) % 97
        ], dtype=float)
        feats.append(vec / 100.0)

    feats = np.stack(feats, axis=0)
    feats = torch.tensor(feats, dtype=torch.float)

    data = from_networkx(G)
    # Ensure we have node features in correct order
    data.x = feats
    # Single-graph batch index (all zeros)
    data.batch = torch.zeros(data.num_nodes, dtype=torch.long)

    return data


### 4) Wrapper: SPARQL Query → PyG Graph

In [5]:
def encode_sparql_query_to_graph(query_str):
    # Parse and translate to algebra
    parsed = parser.parseQuery(query_str)
    alg = algebra.translateQuery(parsed).algebra

    # Algebra → NetworkX
    G = algebra_to_graph(alg)

    # NetworkX → PyG
    pyg_data = graph_to_pyg(G)
    return pyg_data, G

# Quick test with anchor query
data_anchor, G_anchor = encode_sparql_query_to_graph(anchor_query)
print(data_anchor)
print("Graph nodes:", len(G_anchor.nodes), "edges:", len(G_anchor.edges))


Data(edge_index=[2, 6], label=[7], num_nodes=7, x=[7, 3], batch=[7])
Graph nodes: 7 edges: 6


### 6) GCN Encoder

In [7]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_dim=3, hidden=32, out_dim=16):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        # aggregate node embeddings to graph embedding
        x = global_mean_pool(x, batch)
        return x  # shape: [batch_size, out_dim]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNEncoder().to(device)
print(model)

# ------------------------------------------------------------
# 7) Helper: Query → Embedding (with current model)
# ------------------------------------------------------------

def get_embedding(model, query_str):
    data, _ = encode_sparql_query_to_graph(query_str)
    data = data.to(device)
    model.eval()
    with torch.no_grad():
        emb = model(data.x, data.edge_index, data.batch)
    return emb  # shape [1, out_dim]

# Test:
emb_anchor = get_embedding(model, anchor_query)
print("Anchor embedding shape:", emb_anchor.shape)

GCNEncoder(
  (conv1): GCNConv(3, 32)
  (conv2): GCNConv(32, 16)
)


### 8) InfoNCE Contrastive Loss

In [9]:
def info_nce(anchor, positives, negatives, temperature=0.1):
    """
    anchor:     [1, d]
    positives:  list of [1, d]
    negatives:  list of [1, d]
    """
    # Normalize
    anchor = anchor / anchor.norm(dim=-1, keepdim=True)
    pos = torch.cat(positives, dim=0)  # [P, d]
    neg = torch.cat(negatives, dim=0)  # [N, d]

    pos = pos / pos.norm(dim=-1, keepdim=True)
    neg = neg / neg.norm(dim=-1, keepdim=True)

    # Similarity to anchor (dot product)
    pos_sim = (pos @ anchor.T) / temperature   # [P, 1]
    neg_sim = (neg @ anchor.T) / temperature   # [N, 1]

    # Combine logits: first P are positives, then N negatives
    logits = torch.cat([pos_sim, neg_sim], dim=0).T  # [1, P+N]
    # Label index of the correct class = 0..P-1 (we treat "any positive" as target 0; simplest demo)
    # For a simple demo, we mark one representative positive as the target.
    target = torch.tensor([0], dtype=torch.long, device=logits.device)

    loss = F.cross_entropy(logits, target)
    return loss, logits.detach().cpu()


### 9) Single Contrastive Loss Computation (no training yet)


In [10]:

model.eval()

anchor_emb = get_embedding(model, anchor_query)

pos_embs = [get_embedding(model, q) for q in positive_queries]
neg_embs = [get_embedding(model, q) for q in negative_queries]

loss, logits = info_nce(anchor_emb, pos_embs, neg_embs, temperature=0.1)
print("Initial InfoNCE loss (untrained model):", float(loss))
print("Logits:", logits.numpy())


Initial InfoNCE loss (untrained model): 1.3675132989883423
Logits: [[9.998892 9.993329 9.961497 9.966191]]


### 10) Mini Contrastive Training Loop

In [11]:

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20

for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()

    # Recompute embeddings with gradients
    data_anchor, _ = encode_sparql_query_to_graph(anchor_query)
    data_anchor = data_anchor.to(device)

    pos_data = [encode_sparql_query_to_graph(q)[0].to(device) for q in positive_queries]
    neg_data = [encode_sparql_query_to_graph(q)[0].to(device) for q in negative_queries]

    anchor_emb = model(data_anchor.x, data_anchor.edge_index, data_anchor.batch)
    pos_embs   = [model(d.x, d.edge_index, d.batch) for d in pos_data]
    neg_embs   = [model(d.x, d.edge_index, d.batch) for d in neg_data]

    loss, _ = info_nce(anchor_emb, pos_embs, neg_embs, temperature=0.1)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f}")


Epoch 01 | Loss: 1.3675
Epoch 02 | Loss: 1.3655
Epoch 03 | Loss: 1.3633
Epoch 04 | Loss: 1.3609
Epoch 05 | Loss: 1.3582
Epoch 06 | Loss: 1.3554
Epoch 07 | Loss: 1.3522
Epoch 08 | Loss: 1.3489
Epoch 09 | Loss: 1.3453
Epoch 10 | Loss: 1.3413
Epoch 11 | Loss: 1.3369
Epoch 12 | Loss: 1.3320
Epoch 13 | Loss: 1.3265
Epoch 14 | Loss: 1.3204
Epoch 15 | Loss: 1.3139
Epoch 16 | Loss: 1.3069
Epoch 17 | Loss: 1.2991
Epoch 18 | Loss: 1.2904
Epoch 19 | Loss: 1.2807
Epoch 20 | Loss: 1.2700


### 11) Inspect Similarities After Training

In [12]:
model.eval()

anchor_emb = get_embedding(model, anchor_query)
pos_embs   = [get_embedding(model, q) for q in positive_queries]
neg_embs   = [get_embedding(model, q) for q in negative_queries]

def cosine(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)
    return (a @ b.T).item()

print("Cosine similarities to anchor:")
for i, e in enumerate(pos_embs):
    print(f"  Positive {i}: {cosine(anchor_emb, e):.4f}")
for j, e in enumerate(neg_embs):
    print(f"  Negative {j}: {cosine(anchor_emb, e):.4f}")


Cosine similarities to anchor:
  Positive 0: 0.9995
  Positive 1: 0.9970
  Negative 0: 0.9739
  Negative 1: 0.9732
